# Dental Vision V1 — pipeline smoke test\nFast diagnostic run only. Audits annotation geometry, then deliberately overfits 12 labeled images. Do not launch another full training until this passes.\n

In [ ]:
!nvidia-smi\nimport pathlib,shutil,json,zipfile\n%cd /kaggle/working\nshutil.rmtree('/kaggle/working/dental-vision-v1',ignore_errors=True)\n!git clone https://github.com/drhaidarali95/dental-vision-v1.git /kaggle/working/dental-vision-v1\n%cd /kaggle/working/dental-vision-v1\n!pip -q install -r requirements.txt\n

In [ ]:
root=pathlib.Path('data/smoke'); shutil.rmtree(root,ignore_errors=True); root.mkdir(parents=True)\n!python scripts/download_dentex.py --out data/smoke --files training_data.zip\nzpath=root/'training_data.zip'; wanted={'caries','deep caries','periapical lesion','periapical lesions','impacted','impacted tooth','impacted teeth'}; candidates=[]\nwith zipfile.ZipFile(zpath) as z:\n    for name in z.namelist():\n        if not name.lower().endswith('.json'): continue\n        try: d0=json.loads(z.read(name))\n        except Exception: continue\n        if not isinstance(d0,dict) or not {'images','annotations'}.issubset(d0): continue\n        cats=d0.get('categories_3') or d0.get('categories') or []; names={str(c.get('name','')).strip().lower() for c in cats}; score=len(names&wanted); bonus=2 if 'quadrant-enumeration-disease' in name.lower() else 0\n        if score or bonus: candidates.append((score+bonus,len(d0['images']),name,d0,cats))\nassert candidates, 'Disease annotation JSON not found'\n_,_,ann_member,d,cats=max(candidates,key=lambda x:(x[0],x[1])); d['categories']=cats\nfor a in d['annotations']:\n    if 'category_id_3' in a: a['category_id']=a['category_id_3']\nsubset=root/'diagnostic'; (subset/'images').mkdir(parents=True,exist_ok=True)\nwith zipfile.ZipFile(zpath) as z:\n    members=z.namelist()\n    for im in d['images']:\n        fn=str(im['file_name']).replace('\\\\','/').lstrip('./'); matches=[m for m in members if m.endswith('/'+fn) or m==fn] or [m for m in members if pathlib.PurePosixPath(m).name==pathlib.PurePosixPath(fn).name]\n        src=matches[0]; dest=subset/'images'/pathlib.PurePosixPath(fn).name\n        with z.open(src) as r,open(dest,'wb') as w: shutil.copyfileobj(r,w)\n        im['file_name']=dest.name\n(subset/'all.json').write_text(json.dumps(d)); zpath.unlink()\nprint('Prepared',len(d['images']),'images from',ann_member)\n

In [ ]:
!python scripts/audit_detection_geometry.py --images data/smoke/diagnostic/images --annotations data/smoke/diagnostic/all.json --output /kaggle/working/geometry_audit.json\n!python scripts/make_overfit_smoke_subset.py --annotations data/smoke/diagnostic/all.json --output data/smoke/diagnostic/smoke.json --images 12\n

In [ ]:
!python train.py --images data/smoke/diagnostic/images --annotations data/smoke/diagnostic/smoke.json --epochs 80 --batch-size 2 --lr 0.0025 --output /kaggle/working/dentex_smoke_overfit.pt\n!python evaluate.py --images data/smoke/diagnostic/images --annotations data/smoke/diagnostic/smoke.json --checkpoint /kaggle/working/dentex_smoke_overfit.pt --score-threshold 0.001 --output /kaggle/working/smoke_metrics.json\nprint(pathlib.Path('/kaggle/working/smoke_metrics.json').read_text())\nprint('SMOKE TEST COMPLETE')\n